# Customer Churn & Retention Analytics\n\n## 1. Load and inspect

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv("../data/WA_Fn-UseC_-Telco-Customer-Churn.csv")
df.head()


## 2. Structure and data quality

In [ ]:
print("Shape:", df.shape)
df.info()
display(df.describe(include="all").T)


In [ ]:
df.isna().sum().sort_values(ascending=False)


In [ ]:
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")
print("Duplicate customer IDs:", df["customerID"].duplicated().sum())
print("Missing TotalCharges:", df["TotalCharges"].isna().sum())


## 3. Standardize columns

In [ ]:
analysis = df.copy()
analysis["SeniorCitizen"] = analysis["SeniorCitizen"].map({0:"No", 1:"Yes"})
analysis = analysis.rename(columns={
    "customerID":"customer_id", "SeniorCitizen":"senior_citizen",
    "PhoneService":"phone_service", "MultipleLines":"multiple_lines",
    "InternetService":"internet_service", "OnlineSecurity":"online_security",
    "OnlineBackup":"online_backup", "DeviceProtection":"device_protection",
    "TechSupport":"tech_support", "StreamingTV":"streaming_tv",
    "StreamingMovies":"streaming_movies", "PaperlessBilling":"paperless_billing",
    "PaymentMethod":"payment_method", "MonthlyCharges":"monthly_charges",
    "TotalCharges":"total_charges"
})
analysis.columns = [col.lower() for col in analysis.columns]
analysis.head()


## 4. Overall KPIs and segment analysis

In [ ]:
total_customers = analysis["customer_id"].nunique()
churned = (analysis["churn"] == "Yes").sum()
print(f"Total customers: {total_customers:,}")
print(f"Churned customers: {churned:,}")
print(f"Churn rate: {churned/total_customers:.2%}")


In [ ]:
def churn_summary(data, group_col):
    result = data.groupby(group_col).agg(
        customers=("customer_id", "nunique"),
        churned=("churn", lambda s: (s == "Yes").sum())
    )
    result["churn_rate"] = result["churned"] / result["customers"]
    return result.sort_values("churn_rate", ascending=False)

display(churn_summary(analysis, "contract"))
display(churn_summary(analysis, "payment_method"))
display(churn_summary(analysis, "internet_service"))


## 5. Visualizations

In [ ]:
sns.countplot(data=analysis, x="churn")
plt.title("Customer Churn Distribution")
plt.tight_layout()
plt.show()


In [ ]:
rates = churn_summary(analysis, "contract").reset_index()
sns.barplot(data=rates, x="contract", y="churn_rate")
plt.title("Churn Rate by Contract")
plt.ylabel("Churn rate")
plt.tight_layout()
plt.show()


In [ ]:
sns.boxplot(data=analysis, x="churn", y="tenure")
plt.title("Tenure Distribution by Churn Status")
plt.tight_layout()
plt.show()


## 6. Monthly charge exposure

In [ ]:
exposure = analysis.loc[analysis["churn"] == "Yes", "MonthlyCharges"].sum()
print(f"Monthly charge exposure: ${exposure:,.2f}")


## 7. Export cleaned data

In [ ]:
analysis.to_csv("../data/telco_customer_churn_clean.csv", index=False)
